<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Rapid Application Development<br>using Large Language Models</b></h1>
<h2><b>Notebook 99:</b> Deploying Your First NIM</h2>
<br>

In this notebook, we will be loading in a NIM model to run on your GPU-enabled environment. The process is documented extensively in the [NIM "Getting Started" documentation](https://docs.nvidia.com/nim/large-language-models/latest/getting-started.html), so feel free to refer to it if you need any more details. 

<hr>

### **NOTE:** THIS IS ONLY FOR REFERENCE!
**(It also will not work in this environment since it assumes host docker access)**

<hr>

NIMs are packaged as container images on a per model/model family basis. The selected image requires a supported GPU profile and matching driver and NVIDIA Container Toolkit versions. Check the [model-specific prerequisites](https://docs.nvidia.com/nim/large-language-models/2.0.10/get-started/advanced/get-started-nemotron-3.5-lightning.html) before deployment.

### **Kickstarting On Application Startup**

One way to kickstart the environment alongside other microservices is to specify it in the docker-compose file as shown below:

```yaml
  nim:
    ## LLM NIM: Serves the reasoning model from a local NVIDIA runtime
    image: nvcr.io/nim/nvidia/nemotron-3.5-lightning-30b-a3b:2.0.9-variant  ## <-the image you're working with
    runtime: nvidia
    shm_size: 16gb
    ports:
      - 8000:8000
    environment:
      - NGC_API_KEY
      - NIM_MODEL_NAME=nvidia/nemotron-3.5-lightning-30b-a3b
      - NIM_SERVED_MODEL_NAME=nvidia/nemotron-3.5-lightning-30b-a3b
      - NIM_PASSTHROUGH_ARGS=--reasoning-parser nemotron_v3
```

Some NIM images or model branches require an NGC Personal API key. When needed, create one in [NGC API Keys](https://org.ngc.nvidia.com/setup/api-keys) and supply it through the host's `NGC_API_KEY` environment variable. For the feature-branch example in the linked guide, this key is optional. Accept the model's governing terms in the [NGC catalog](https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/nemotron-3.5-lightning-30b-a3b) before the first download.

In [ ]:
import os

In [ ]:
%%bash
set -e
if [ -n "${NGC_API_KEY:-}" ]; then
    printf '%s' "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin
fi

<br>

After this, you should be able to select your models of choice from the [**NGC Catalog**](https://catalog.ngc.nvidia.com/containers?filters=nvidia_nim). The code block below specifies the model of interest and then proceeds to actually download the NIM, download the associated model, and kickstart the microservice with docker.

In [ ]:
# Choose a LLM NIM Image from NGC
%env IMG_NAME=nvcr.io/nim/nvidia/nemotron-3.5-lightning-30b-a3b:2.0.9-variant

# Choose a path on your system to cache the downloaded models
os.environ["LOCAL_NIM_CACHE"] = os.path.abspath("./cache/nim")

!mkdir -p "$LOCAL_NIM_CACHE"

## Start the LLM NIM
# !docker rm -f nim
!docker run --rm --name=nim \
    --runtime=nvidia \
    --gpus all \
    --shm-size=16GB \
    --network nvidia-rad \
    -e NGC_API_KEY \
    -e NIM_MODEL_NAME=nvidia/nemotron-3.5-lightning-30b-a3b \
    -e NIM_SERVED_MODEL_NAME=nvidia/nemotron-3.5-lightning-30b-a3b \
    -e 'NIM_PASSTHROUGH_ARGS=--reasoning-parser nemotron_v3' \
    -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
    -u "$(id -u):0" \
    -p 8000:8000 \
    "$IMG_NAME"

## NOTES:
## - gpus, runtime, and shared-memory-size all help specify which resources 
##     and environments the service can leverage. 
## - Since we put our microservices (including our jupyter labs instance)
##     on the nvidia-rad network in the dockerfile, specifying this helps
##     the services play nicely together. This detail is unimportant.
## - -e specifies environment variables, and NIMs want access to NGC.
## - -v specifies volume mounts, and we may want access to the NIM model cache
##     to avoid redownloading resources and maybe modify some settings.
## - -u specifies user profile and -p specifies port mapping. We usually 
##     want parity with our other services to keep things consistent.
## - $IMG_NAME AKA nvcr.io/nim/<model-name> here is the image we run

<br><hr>

#### **AFTER THE STARTUP MESSAGES, CHECK READINESS:**
```json
INFO ... on.py:62] Application startup complete.
INFO ... server.py:214] Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO...8 metrics.py:334] Avg prompt throughput: 0.3 tokens/s, Avg generation throughput: 1.5 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%
```

In another terminal on the same host, check readiness and confirm that `/v1/models` lists `nvidia/nemotron-3.5-lightning-30b-a3b`:

```bash
curl --fail http://localhost:8000/v1/health/ready
curl --fail http://localhost:8000/v1/models
```

<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
